In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import edunn as nn
from edunn import utils

# Gradient descent

Gradient descent is a simple but effective optimization technique for training differentiable models.

At every iteration of the algorithm we compute the derivative of the error with respect to each of the parameters, `δEδp`, and update the weights in the direction opposite to the gradient. This update is mediated by the parameter `α`, which sets the learning rate.

The gradient descent algorithm is simple:

```python
for i in range(iterations):
    for p in model.parameters()
        # we use p[:] to modify the values of p
        # instead of creating a new variable
        p[:] = p - α * δEδp(x,y)
```

This pseudocode leaves out some cumbersome parts. In particular, the iteration over the input values `x` and output values `y` of the examples, in its `batches` version, and the computation of the error and of the derivatives `δEδp`.

The `edunn` library provides the `BatchedGradientOptimizer` class, which takes care of all that and lets us implement an optimizer very easily by creating a subclass of it and implementing the `optimize_batch` method, where we only have to worry about optimizing the model using the derivatives computed from a batch of the dataset.

For this exercise we have created the `GradientDescent` class, which subclasses `BatchedGradientOptimizer`. Now implement the crucial part of the `optimize_batch` method of `GradientDescent`, so that it updates the parameters based on the gradients that have already been computed.

To test this optimizer, we are going to use a fake model and a fake error function that let us control the input to the optimizer. The flexibility of `edunn`'s `Model` class makes this very easy to do by creating the `FakeModel` and `FakeError` classes, which actually ignore their inputs and outputs and exist only so that `FakeModel` initializes 2 parameters with value 0 and returns `[-1,1]` as their derivative.

In [ ]:
#Fake model with a parameter vector whose initial value is [0,0] and whose gradients are always [1,-1]
model = nn.FakeModel(parameter=np.array([0,0]),gradient=np.array([1, -1]))
# fake error function whose error is always 1, and so are its derivatives
error = nn.FakeError(error=1,derivative_value=1)

# Fake dataset, which will not actually be used
fake_samples = 3
fake_x = np.random.rand(fake_samples,10)
fake_y = np.random.rand(fake_samples,5)

# Optimize the model for 1 epoch with lr=2
optimizer = nn.GradientDescent(batch_size=fake_samples,epochs=1,lr=2,shuffle=False)
history = optimizer.optimize(model,fake_x,fake_y,error,verbose=False)
expected_parameters=np.array([-2,2])
utils.check_same(expected_parameters,model.get_parameters()["parameter"])

# Optimize the model for 1 *additional* epoch with lr=2
history = optimizer.optimize(model,fake_x,fake_y,error,verbose=False)
expected_parameters=np.array([-4,4])
utils.check_same(expected_parameters,model.get_parameters()["parameter"])
    
# Optimize the model for 3 more epochs, now with lr=1    
optimizer = nn.GradientDescent(batch_size=fake_samples,epochs=3,lr=1,shuffle=False)
history = optimizer.optimize(model,fake_x,fake_y,error,verbose=False)
expected_parameters=np.array([-7,7])
utils.check_same(expected_parameters,model.get_parameters()["parameter"])    
    

# Training a Linear Regression model with gradient descent

Now that we have all the pieces, we can define and train our first `LinearRegression` model to estimate house prices using the [Boston Housing](https://www.kaggle.com/c/boston-housing) dataset

In [ ]:
import edunn as nn
import numpy as np
from edunn import metrics,datasets

x,y=datasets.load_regression("boston")
x = (x-x.mean(axis=0))/x.std(axis=0)
n, din = x.shape
n, dout = y.shape
print("Dataset sizes:", x.shape,y.shape)

#Linear regression model
model = nn.LinearRegression(din,dout)
error = nn.MeanError(nn.SquaredError())
optimizer = nn.GradientDescent(lr=0.001,epochs=1000,batch_size=32)

# Optimization algorithm
history = optimizer.optimize(model,x,y,error)
nn.plot.plot_history(history,error_name=error.name)


print("Model error:")
y_pred=model.forward(x)
metrics.regression_summary(y,y_pred)
nn.plot.regression1d_predictions(y,y_pred)

# Comparison with sklearn

As an additional check, we will compute the optimal parameters of a linear regression model with sklearn and visualize the results. The error should be similar to that of our model (RMSE=4.68).

In [ ]:
from sklearn import linear_model
model=linear_model.LinearRegression()
model.fit(x,y)
y_pred=model.predict(x)
print("Model error:")
metrics.regression_summary(y,y_pred)
print()

nn.plot.regression1d_predictions(y,y_pred)
